# We will need to process the CHROMA dataset now

Here is what we know:

- Just like before we have 2 datasets recorded at different times that need to be stiched together
- This time, **THE FEATURES ARE SEPARATED BY NUCLEAR AND CTYOPLASM**
    - **these need to be concatenated horizontally**
- Furthermore, Nuclear datasets are split into *4* different nuclear properties/features

### What we need to do:

1. Format all OBJ files
2. concatenate all features horizontally
3. vertically concatenate / stitch both datasets + adjust timepoints

In [1]:
import polars as pl
from pathlib import Path
import unicodedata
import re
import os
import shutil
from time import sleep
from datetime import datetime
from collections import defaultdict
from functools import reduce

drop_cols = [
    "Row",
    "Column",
    "Field",
    "Plane",
    "Object No",
    "X",
    "Y",
    "Bounding Box",
    "Position X [um]",
    "Position Y [um]",
    "Compound",
    "Concentration",
    "Cell Type",
    "Cell Count"
]

def clean_folder(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
    os.makedirs(path, exist_ok=True)

def clean_column(name):
    replacements = {
        'µ': 'u',
        'μ': 'u',
        '²': '2',
        '³': '3',
    }
    
    for char, replacement in replacements.items():
        name = name.replace(char, replacement)
    
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("ascii")
    name = name.strip()
    name = name.strip('_')
    
    return name

def parse_harmony_dataset(root: Path) -> tuple[Path, str]:
    """
    Parses a harmony file directory and yields the cell analysis file

    Args:
        root (Path): Top level folder where all the harmony datasets are located

    Yields:
        Path, str: yields a file path to the cell analysis AND the harmony dataset name
    
    **NOTE:** The harmony file structure is:
        
        raw name/
        ├─ Evaluation/
        │  ├─ PlateResult.txt
        │  ├─ Objects_Population - NoBorderObj.txt (this is analysis file)
        │  ├─ .ExportDone_CellListsToTxt
        ├─ indexfile.txt

    """
    
    for f in root.glob("*/*/*"):
        if f.is_file() and not f.name.startswith(".") and f.name != "PlateResults.txt":
            # print(f)
            yield f, f.relative_to(root).parts[0]

In [2]:
raw_dataset_path = Path("/home/derfelt/PartchLabFiles/Circadian "
                        "Rhythms/Live Study/May-Harmony-CHROMA-datasets")

cytoplasm_raw_path = raw_dataset_path / "cytoplasm"
nuclear_raw_path = raw_dataset_path / "nuclear"

## Let's work on the nuclear features first

In [3]:
root_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                        "Study/May-Harmony-CHROMA-datasets")
temp_dir_nuclear = root_dir / f'.{nuclear_raw_path.stem}_TEMP'
clean_folder(temp_dir_nuclear)
for i, (f_path, dataset_name) in enumerate(parse_harmony_dataset(nuclear_raw_path)):
    print(dataset_name, i)
    print(f_path.stem)
    
    out_name = temp_dir_nuclear / f"{f_path.stem}._.{dataset_name}.tsv"
    with (open(f_path, "r", encoding="utf-8") as in_file, 
          open(out_name, "w", encoding='utf-8') as out_file):
        for i, line in enumerate(in_file):
            curr_line = line.strip()
            
            if i < 9:
                print(f"{curr_line} :: {i}")
                continue
            if i == 9:
                cleaned = "\t".join(clean_column(col) for col in line.strip().split("\t"))
                out_file.write(cleaned + "\n")
            else:
                out_file.write(line.rstrip("\t\n") + "\n")
    
    print("\n\n")

20260507_Bev_restartLiveCP_cytoClock__2026-05-07T19_18_39-Dataset 1 0
Objects_Population - DimNucleiRound
Database Name	ODA :: 0
Database Link	http://localhost/ODA/OdaService.asmx :: 1
Evaluation Signature	9cf20827-c6ba-49e6-9116-68422c93ebc8 :: 2
Plate Name	20260507_Bev_restartLiveCP_cytoClock :: 3
Measurement	Measurement 2 :: 4
Evaluation	Evaluation6 :: 5
Population	Population - DimNucleiRound :: 6
 :: 7
[Data] :: 8



20260507_Bev_restartLiveCP_cytoClock__2026-05-07T19_18_39-Dataset 1 1
Objects_Population - NoBorderDimNucRound
Database Name	ODA :: 0
Database Link	http://localhost/ODA/OdaService.asmx :: 1
Evaluation Signature	9cf20827-c6ba-49e6-9116-68422c93ebc8 :: 2
Plate Name	20260507_Bev_restartLiveCP_cytoClock :: 3
Measurement	Measurement 2 :: 4
Evaluation	Evaluation6 :: 5
Population	Population - NoBorderDimNucRound :: 6
 :: 7
[Data] :: 8



20260507_Bev_restartLiveCP_cytoClock__2026-05-07T19_18_39-Dataset 1 2
Objects_Population - NoBorderRegNuc
Database Name	ODA :: 0
Database Li

In [23]:
shutil.rmtree(temp_dir_nuclear)

# As of right now, the nuclear features are impossible to merge

Here's the strat:
1. vertically concat everything
2. calculate the timepoints for each one
3. preprocess/aggregate
4. horizontally merge each one

**This will be a pain in the ass**

## In the meantime, I will process the chroma dataset

In [9]:
cytoplasm_raw_path

root_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/May-Harmony-CHROMA-datasets")
temp_dir_cyto = root_dir / ".CYTO-TEMP/"
clean_folder(temp_dir_cyto)

for f, dataset_name in parse_harmony_dataset(cytoplasm_raw_path):
    print(dataset_name, f)
    
    out_name = temp_dir_cyto / f"{dataset_name}._.TEMP_FORMAT.tsv"
    
    with (open(f, "r", encoding="utf-8") as in_file, 
          open(out_name, "w", encoding='utf-8') as out_file):
        for i, line in enumerate(in_file):
            curr_line = line.strip()
            
            if i < 9:
                print(f"{curr_line} :: {i}")
                continue
            if i == 9:
                cleaned = "\t".join(clean_column(col) for col in line.strip().split("\t"))
                out_file.write(cleaned + "\n")
            else:
                out_file.write(line.rstrip("\t\n") + "\n") 
    
    lf = (
        pl
        .scan_csv(out_name, 
                    separator='\t', 
                    null_values=["", "NA", "NaN", "nan", "NULL"])
        .with_columns(
            (
                pl.col("Row").map_elements(lambda x: chr(64 + x), return_dtype=pl.String) +
                pl.col("Column").cast(pl.String)
            ).alias("WellName")
        )
        .select(
            ["WellName", pl.all().exclude("WellName")]
        )
        .drop(drop_cols)
        )
    
    lf.sink_csv(str(out_name).replace("._.TEMP_FORMAT", ""), separator="\t")
    
    out_name.unlink(missing_ok=True)
    
            
    print("\n\n")

20260507_Bev_restartLiveCP_cytoClock__2026-05-07T19_18_39-Dataset 1 /home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/May-Harmony-CHROMA-datasets/cytoplasm/20260507_Bev_restartLiveCP_cytoClock__2026-05-07T19_18_39-Dataset 1/Evaluation5/Objects_Population - Cells-001.txt
Database Name	ODA :: 0
Database Link	http://localhost/ODA/OdaService.asmx :: 1
Evaluation Signature	7ac4d21d-1ea3-4293-8f4d-1b2f91f04a5d :: 2
Plate Name	20260507_Bev_restartLiveCP_cytoClock :: 3
Measurement	Measurement 2 :: 4
Evaluation	Evaluation5 :: 5
Population	Population - Cells :: 6
 :: 7
[Data] :: 8



20260511_Bev_restartLiveCP_cytoClock__2026-05-11T14_42_49-Dataset 2 /home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/May-Harmony-CHROMA-datasets/cytoplasm/20260511_Bev_restartLiveCP_cytoClock__2026-05-11T14_42_49-Dataset 2/Evaluation3/Objects_Population - Cells-001.txt
Database Name	ODA :: 0
Database Link	http://localhost/ODA/OdaService.asmx :: 1
Evaluation Signature	88f51d04-4e7e-41d2-b026-b3bd0d2f

### lets convert the timepoints

In [17]:
def extract_t0(f: Path):
    dt_str = f.stem.split("__")[-1].split("-Dataset")[0]
    return datetime.strptime(dt_str, "%Y-%m-%dT%H_%M_%S")

out_dir = Path("/home/derfelt/PartchLabFiles/Circadian "
               "Rhythms/Live Study/analysis_ready_data")

sampling_interval = 1.5
timepoint_col = "Timepoint"

dataset_files = sorted(temp_dir_cyto.iterdir(), key= lambda k: int(str(k.stem).split("Dataset ")[-1]))

t0s: list[datetime] = [extract_t0(f) for f in dataset_files]

lfs = []
for i, (f, t0) in enumerate(zip(dataset_files, t0s)):
    offset = (t0 - t0s[0]).total_seconds() / 3600
    offset = round(offset / sampling_interval) * sampling_interval
    
    lf = (
        pl.scan_csv(f, separator="\t")
        .with_columns(
            (offset + (pl.col(timepoint_col) - 1) * sampling_interval)
            .alias(timepoint_col)
        )
    )
    lfs.append(lf)
    print(f"Dataset {i + 1}: t0={t0}, offset={offset:.2f}h")
    
concat_lf = pl.concat(lfs)

out_file = out_dir / f"2026-05-11_Harmony_CHROMA-STITCHED._.CYTO-ONLY.tsv"

concat_lf.sink_csv(out_file, separator="\t")


Dataset 1: t0=2026-05-07 19:18:39, offset=0.00h
Dataset 2: t0=2026-05-11 14:42:49, offset=91.50h
